# Embeddings: modelo de lenguaje con contexto de 3 caracteres (estilo Bengio et al., 2003)

**Problema del one-hot:** para mirar más contexto, concatenar one-hots crece rápido (3 caracteres = 120 entradas) y cada caracter es "igual de distinto" a todos los demás: la red no puede aprovechar que, por ejemplo, las vocales se comportan parecido.

**Embeddings:** a cada caracter le asignamos un vector **chico y aprendible** de números reales (acá de 2 dimensiones). Esos vectores se guardan en una matriz `C` de 40x2: la fila `i` es el embedding del caracter `i`. Durante el entrenamiento los embeddings se ajustan, y caracteres que se usan en contextos parecidos terminan con vectores parecidos.

Buscar el embedding de un caracter, `C[i]`, es lo mismo que `one_hot(i) @ C`: un embedding es una capa lineal aplicada a un one-hot, pero calculada de forma eficiente.

### Apuntes de clase
Se concatenan los embeddings. Si cada embedding tiene 2 entradas, la matriz C va a ser 40x2. Por ejemplo, si `*la` es igual a 0 25 14 (los índices de `*`, `l` y `a`), concatena los embeddings correspondientes a cada número. Las dimensiones de esta concatenación quedan 1x6, que vamos a enviar a una red neuronal con 6 entradas y 40 salidas. Ese arreglo es de log counts. Serían 40 neuronas con 6 entradas.

La matriz de los pesos queda 6x40. Si se multiplica (1x6) @ (6x40), queda 1x40: un logit para cada uno de los 40 caracteres posibles.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Cada línea del archivo es un nombre de ciudad (ya codificado en punycode). Descartamos líneas vacías.
dataset = [d for d in open('data/city_names_puny.txt', 'r').read().split('\n') if d]

# Vocabulario: '*' marca inicio/fin de palabra y va en el índice 0, después todos los caracteres ordenados
charset = ['*'] + sorted(list(set([c for d in dataset for c in d])))
ctoi = {c: i for i, c in enumerate(charset)}   # caracter -> índice
itoc = {i: c for i, c in enumerate(charset)}   # índice -> caracter
nclasses = len(charset)                        # 40
len(dataset), nclasses

## Construcción del dataset
Vamos a armar un dataset para entrenar una red neuronal similar a la de Y. Bengio. El modelo mira **3 caracteres en el pasado** y predice **1 caracter en el futuro**.

* El contexto arranca como `***` (índices `[0, 0, 0]`).
* Por cada caracter `c` de la palabra (más el `*` final), el ejemplo es `contexto -> c`.
* Después **corremos la ventana**: sacamos el caracter más viejo del contexto y agregamos `c`.

In [ ]:
context_size = 3   # el tamaño del contexto: cuántos caracteres usamos para predecir el siguiente

def build_dataset(palabras, verbose=False):
    X, Y = [], []
    for d in palabras:   # por cada palabra en el dataset
        if verbose:
            print(d)
        example = list(d) + ['*']
        context = [0] * context_size   # en principio, '***'
        for c in example:   # por cada caracter en example
            if verbose:
                print(''.join([itoc[x] for x in context]) + ' --> ' + c)
            X.append(context)   # el contexto actual
            Y.append(ctoi[c])   # el índice del caracter que sigue
            context = context[1:] + [ctoi[c]]   # corremos la ventana (adentro del loop, así cambia en cada caracter)
    return torch.tensor(X), torch.tensor(Y)

build_dataset(['la plata'], verbose=True);   # ejemplo chico para ver cómo se arma

In [ ]:
X, Y = build_dataset(dataset)   # dataset completo (sin print: son más de 1,7 millones de ejemplos)
X.shape, Y.shape                # X: (N, 3) índices de contexto, Y: (N,) índice del caracter siguiente

## El modelo
1. **Embedding:** `C[X]` busca el embedding de cada uno de los 3 caracteres: (N, 3) -> (N, 3, 2).
2. **Concatenación:** `.view(-1, 6)` pone los 3 embeddings uno al lado del otro: (N, 6).
3. **Capa lineal:** `emb @ W + b`, con W de 6x40 y b de 40: (N, 40) logits (log counts).
4. **Softmax + NLL:** igual que en bigramas (`F.cross_entropy` hace las dos cosas).

Parámetros: `C` (40x2 = 80), `W` (6x40 = 240) y `b` (40). En total, solo **360 números**.

In [ ]:
g = torch.Generator().manual_seed(42)
emb_dim = 2
C = torch.randn((nclasses, emb_dim), generator=g)                  # embeddings: 40x2
W = torch.randn((context_size * emb_dim, nclasses), generator=g)   # pesos: 6x40
b = torch.randn(nclasses, generator=g)                             # sesgo: 40
parameters = [C, W, b]
for p in parameters:
    p.requires_grad = True
print('parámetros:', sum(p.nelement() for p in parameters))

# Un forward de ejemplo con los primeros 5 contextos, para ver las dimensiones
emb = C[X[:5]]                                      # (5, 3, 2)
logits = emb.view(-1, context_size * emb_dim) @ W + b   # (5, 6) @ (6, 40) + (40) = (5, 40)
emb.shape, logits.shape

**Sobre `view`:** `C[X]` devuelve un tensor de (N, 3, 2): para cada ejemplo, 3 embeddings de 2 números. `view(-1, 6)` **reinterpreta** esos mismos números como (N, 6), o sea los 3 embeddings uno al lado del otro: eso es la concatenación. No copia datos (es muy eficiente), y el `-1` le dice a PyTorch "calculá vos esta dimensión" (acá, N).

In [ ]:
emb = C[X[:1]]                 # el primer ejemplo: contexto '***'
print(emb)                     # (1, 3, 2)
print(emb.view(-1, 6))         # (1, 6): los 3 embeddings concatenados

## Entrenamiento con minibatches (SGD)
Igual que antes: en cada paso tomamos 512 ejemplos al azar, hacemos forward, backward y update. En el último 25% bajamos el learning rate (*learning rate decay*) para afinar.

In [ ]:
def nll_completa(X, Y, chunk=100_000):
    total = 0.0
    with torch.no_grad():
        for i in range(0, len(Y), chunk):
            logits = C[X[i:i+chunk]].view(-1, context_size * emb_dim) @ W + b
            total += F.cross_entropy(logits, Y[i:i+chunk], reduction='sum').item()
    return total / len(Y)

steps, batch_size, lr = 20_000, 512, 0.5
lossi = []
for k in range(steps):
    ix = torch.randint(0, X.shape[0], (batch_size,), generator=g)
    # forward pass
    emb = C[X[ix]]
    logits = emb.view(-1, context_size * emb_dim) @ W + b
    loss = F.cross_entropy(logits, Y[ix])
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    paso = lr if k < steps * 0.75 else lr / 10
    for p in parameters:
        p.data += -paso * p.grad
    lossi.append(loss.item())
    if k % 2000 == 0:
        print(k, loss.item())

print(f'NLL (dataset completo): {nll_completa(X, Y):.4f}')

In [ ]:
plt.plot(torch.tensor(lossi).view(-1, 100).mean(1))   # loss promedio cada 100 pasos
plt.xlabel('paso (x100)'); plt.ylabel('NLL minibatch'); plt.grid();

## Visualizando los embeddings
Como cada embedding tiene 2 dimensiones, podemos dibujarlos en un plano. Caracteres que el modelo usa de forma parecida (por ejemplo las vocales, o los dígitos) tienden a quedar cerca.

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(C[:, 0].data, C[:, 1].data, s=200)
for i in range(nclasses):
    plt.text(C[i, 0].item(), C[i, 1].item(), itoc[i], ha='center', va='center', color='white')
plt.grid();

### ¿Por qué no mejora respecto de bigramas?
La NLL queda en ~2,72, igual que el modelo de bigramas, aunque ahora miramos 3 caracteres. Hay dos cuellos de botella:
* **Embeddings de 2 dimensiones:** cada caracter se resume en solo 2 números, y los 40 caracteres tienen que "entrar" en un plano. Se pierde mucha información sobre cuál es cada caracter.
* **Modelo lineal:** los logits son `emb @ W + b`, o sea una **suma** de aportes independientes de cada posición del contexto. El modelo no puede aprender combinaciones del tipo "si los dos últimos son `xn`, entonces viene `-`".

Probamos las dos mejoras por separado.

## Mejorando el modelo
Generalizamos el entrenamiento a una función con:
* `emb_dim`: tamaño del embedding de cada caracter.
* `hidden`: cantidad de neuronas de una **capa oculta** con activación `tanh` (0 = sin capa oculta, como el modelo de clase).

Con capa oculta el modelo es el de **Bengio et al. (2003)**: embeddings -> concatenación -> `tanh(emb @ W1 + b1)` -> `h @ W2 + b2` -> softmax.
* La **no linealidad** (`tanh`) permite que la red combine la información de los 3 caracteres en lugar de solo sumarla.
* Los pesos de las capas los inicializamos **chicos** (multiplicados por 0,2 y 0,1): si son grandes, `tanh` se satura (queda en ±1), su gradiente $1 - \tanh^2$ es casi 0 y la red no aprende.

In [ ]:
def forward(params, Xb):
    C, W1, b1, W2, b2 = params
    h = C[Xb].view(-1, context_size * C.shape[1])   # embeddings concatenados
    if W1 is not None:
        h = torch.tanh(h @ W1 + b1)                  # capa oculta
    return h @ W2 + b2                               # logits

def entrenar(emb_dim, hidden=0, lr=0.5, steps=20_000, batch_size=512):
    g = torch.Generator().manual_seed(42)
    C = torch.randn((nclasses, emb_dim), generator=g)
    if hidden:
        W1 = torch.randn((context_size * emb_dim, hidden), generator=g) * 0.2
        b1 = torch.zeros(hidden)
        W2 = torch.randn((hidden, nclasses), generator=g) * 0.1
    else:
        W1 = b1 = None
        W2 = torch.randn((context_size * emb_dim, nclasses), generator=g)
    b2 = torch.zeros(nclasses)
    params = [C, W1, b1, W2, b2]
    entrenables = [p for p in params if p is not None]
    for p in entrenables:
        p.requires_grad = True

    for k in range(steps):
        ix = torch.randint(0, X.shape[0], (batch_size,), generator=g)
        loss = F.cross_entropy(forward(params, X[ix]), Y[ix])
        for p in entrenables:
            p.grad = None
        loss.backward()
        paso = lr if k < steps * 0.75 else lr / 10
        for p in entrenables:
            p.data += -paso * p.grad

    with torch.no_grad():
        total = sum(F.cross_entropy(forward(params, X[i:i+100_000]), Y[i:i+100_000], reduction='sum').item()
                    for i in range(0, len(Y), 100_000))
    nll = total / len(Y)
    print(f'emb_dim={emb_dim:2d} hidden={hidden:3d} parámetros={sum(p.nelement() for p in entrenables):6d}  NLL={nll:.4f}')
    return params, nll

In [ ]:
params_emb10, nll_emb10 = entrenar(emb_dim=10, hidden=0, lr=0.5)       # embeddings más grandes, sin capa oculta
params_bengio, nll_bengio = entrenar(emb_dim=10, hidden=100, lr=0.1)   # modelo de Bengio: embeddings + capa oculta tanh

In [ ]:
print(f'Bigramas (conteo):                    2.7188')
print(f'Embeddings 2, lineal (modelo clase):  {nll_completa(X, Y):.4f}')
print(f'Trigramas one-hot (trigramas_nn):     2.5744')
print(f'Embeddings 10, lineal:                {nll_emb10:.4f}')
print(f'Embeddings 10 + capa oculta (Bengio): {nll_bengio:.4f}')

## Sampleando del modelo de Bengio
Empezamos con el contexto `***`, la red da la distribución del siguiente caracter, sampleamos y corremos la ventana del contexto hasta que salga `*`.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(15):
    out = []
    context = [0] * context_size
    while True:
        logits = forward(params_bengio, torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itoc[ix])
        context = context[1:] + [ix]
    print(''.join(out))

## Conclusiones
* Con embeddings podemos usar **más contexto** con **pocos parámetros**: el modelo de clase tiene 360 y el de Bengio unos 7.500, contra los 40^4 = 2,5 millones de una tabla de 4-gramas por conteo.
* Los embeddings se **aprenden** junto con el resto de la red y ubican cerca a caracteres que se comportan parecido.
* Con embeddings de 2 dimensiones y un modelo lineal, la NLL no mejora respecto de bigramas (~2,72): hay muy poca capacidad.
* Agrandar los embeddings (10 dimensiones) ya mejora a ~2,56, parecido al trigrama one-hot.
* Agregar la **capa oculta con `tanh`** (modelo de Bengio) da la mejor NLL (~2,41): la no linealidad permite combinar los caracteres del contexto.